# AvitoTech ML CUP 2026 — Neural-Only Submission Notebook

Run all cells in Kaggle. The notebook trains exactly one end-to-end neural recommender and writes `submission.csv` in the current working directory.

Design for RAM safety on Kaggle 2×T4:
- never loads the 5B-event train clickstream into memory;
- reads train/eval parquet files in batches;
- stores only capped next-contact training examples;
- maps `item_id` with vectorized `np.searchsorted`, not a giant Python dict;
- uses item feature selection from the official schema: vertical/category/geo/semantic ids;
- builds a non-trainable candidate set, then ranks candidates only with the neural network.

In [ ]:
# =========================
# 0. CONFIG: edit here only
# =========================
from pathlib import Path

DATA_ROOT = Path('/kaggle/input/datasets/nikitakuznetsof/avito-ml-cup-2026')
SUBMISSION_FILE = 'submission.csv'  # no root path, as requested
MODEL_DIR = Path('model_artifacts')
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR = Path('reports')
REPORT_DIR.mkdir(exist_ok=True)

# Real data sizes from ODS: item_features 1.85GB, eval history 705MB zipped, train zips ~38.7GB, 94,408 eval users.
# These defaults keep RAM below typical Kaggle 2xT4 limits while still training on real clickstream signal.
SEED = 42
MAX_SEQ_LEN = 48
MAX_TRAIN_POSITIVES = 900_000       # capped arrays: roughly 270MB for seq tensors before dataloader overhead
TRAIN_PARTITION_LIMIT = 20          # train_data partitions to scan; raise to 40/60 if time allows
TRAIN_BATCH_ROWS = 180_000
EVAL_BATCH_ROWS = 220_000

# Model/training. 2xT4 notebook usually exposes cuda:0; DataParallel is optional and not needed for stability.
BATCH_SIZE = 768
EPOCHS = 4
NEGATIVES = 48
EMB_DIM = 112
HIDDEN_DIM = 176
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2
USE_DATA_PARALLEL = False

# Candidate generation: final ordering is neural; candidate pool is only to avoid scoring millions of items per user.
GLOBAL_CANDIDATES = 900
PER_FEATURE_CANDIDATES = 180
PER_USER_MAX_CANDIDATES = 4096
SUBMISSION_K = 160

# Official eval excludes verticals 1 and 6 in the public popular baseline / eval bucket logic.
VALID_VERTICALS = {0, 2, 3, 4, 5, 7}

print('DATA_ROOT:', DATA_ROOT)
print('SUBMISSION_FILE:', SUBMISSION_FILE)


In [ ]:
# =========================
# 1. Imports and environment
# =========================
import gc
import json
import os
import random
from collections import Counter, defaultdict, deque
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm.auto import tqdm

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)
print('cuda devices:', torch.cuda.device_count())
if DEVICE.type == 'cuda':
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


In [ ]:
# =========================
# 2. Paths and official schema columns
# =========================
def must_exist(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def find_first(patterns: Iterable[str]) -> Path:
    for pattern in patterns:
        hits = sorted(DATA_ROOT.glob(pattern))
        if hits:
            return hits[0]
    raise FileNotFoundError(f'Could not find any of: {list(patterns)}')

ITEM_ID_COL = 'item_id'
EVENT_COLUMNS = ['timestamp', 'eid', 'user_id', 'item_id']
ITEM_FEATURE_COLS = [
    'vertical_id',       # 8 catalog verticals
    'category_ext_y',   # low-level taxonomy
    'region_id_y',      # region
    'loc_id_y',         # local geo
    'sid_0_y', 'sid_1_y', 'sid_2_y', 'sid_3_y',  # RQ-BERT semantic ids
]

ITEMS_PATH = must_exist(DATA_ROOT / 'item_features.parquet')
EVAL_USERS_PATH = must_exist(DATA_ROOT / 'eval_users.csv')
CONTACT_EIDS_PATH = must_exist(DATA_ROOT / 'contact_eids.csv')
EVAL_EVENTS_PATH = find_first(['eval_user_events/eval_user_events.pq', 'eval_user_events.pq', 'eval_user_events.parquet'])
TRAIN_PATHS = sorted(DATA_ROOT.glob('train_*/*/part_*.parquet')) or sorted(DATA_ROOT.glob('train_data/part_*.parquet'))
TRAIN_PATHS = TRAIN_PATHS[:TRAIN_PARTITION_LIMIT]

print('ITEMS_PATH:', ITEMS_PATH)
print('EVAL_EVENTS_PATH:', EVAL_EVENTS_PATH)
print('train partitions selected:', len(TRAIN_PATHS))
print('first train paths:', [str(p) for p in TRAIN_PATHS[:3]])


In [ ]:
# =========================
# 3. Item feature selection and compact encoding
# =========================
schema_names = pq.ParquetFile(ITEMS_PATH).schema.names
feature_cols = [c for c in ITEM_FEATURE_COLS if c in schema_names]
if len(feature_cols) < 4:
    raise RuntimeError(f'Unexpected item feature schema. Found columns: {schema_names[:20]}')

# Read only selected low-cardinality columns, not the full 1.85GB payload.
items = pd.read_parquet(ITEMS_PATH, columns=[ITEM_ID_COL] + feature_cols)
items = items.drop_duplicates(ITEM_ID_COL).reset_index(drop=True)
items[ITEM_ID_COL] = items[ITEM_ID_COL].astype('int64')
print('items rows:', len(items))
print('selected feature columns:', feature_cols)

item_ids = items[ITEM_ID_COL].to_numpy(np.int64)
row_to_item_id = np.empty(len(item_ids) + 1, dtype=np.int64)
row_to_item_id[0] = -1
row_to_item_id[1:] = item_ids

# Vectorized item_id -> row mapping. Avoids multi-GB Python dict overhead on large catalog.
sort_order = np.argsort(item_ids)
sorted_item_ids = item_ids[sort_order]
sorted_rows = (np.arange(len(item_ids), dtype=np.int32) + 1)[sort_order]

def map_item_ids_to_rows(values) -> np.ndarray:
    arr = np.asarray(values, dtype=np.int64)
    idx = np.searchsorted(sorted_item_ids, arr)
    valid = idx < len(sorted_item_ids)
    if valid.any():
        check = valid.copy()
        check[valid] = sorted_item_ids[idx[valid]] == arr[valid]
        valid = check
    out = np.zeros(len(arr), dtype=np.int32)
    if valid.any():
        out[valid] = sorted_rows[idx[valid]]
    return out

feature_matrix = np.zeros((len(items) + 1, len(feature_cols)), dtype=np.int32)
raw_feature_arrays = {}
cardinalities = []
for j, col in enumerate(feature_cols):
    s = items[col]
    fill = -1 if pd.api.types.is_numeric_dtype(s) else '__NA__'
    codes, uniques = pd.factorize(s.fillna(fill), sort=False)
    encoded = codes.astype(np.int32) + 1
    feature_matrix[1:, j] = encoded
    raw_feature_arrays[col] = encoded.copy()  # row-1 indexed; compact int32 for candidate maps
    cardinalities.append(int(len(uniques)))
    del s, codes, uniques, encoded

vertical_raw = items['vertical_id'].fillna(-1).astype('int16').to_numpy() if 'vertical_id' in items.columns else None
valid_item_mask = np.ones(len(item_ids) + 1, dtype=bool)
valid_item_mask[0] = False
if vertical_raw is not None:
    valid_item_mask[1:] = np.isin(vertical_raw, list(VALID_VERTICALS))

print('feature_matrix shape:', feature_matrix.shape, 'MB:', round(feature_matrix.nbytes / 1024**2, 1))
print('cardinalities:', dict(zip(feature_cols, cardinalities)))
del items
gc.collect()


In [ ]:
# =========================
# 4. Eval users and contact event ids
# =========================
eval_users_df = pd.read_csv(EVAL_USERS_PATH)
USER_COL = 'user_id' if 'user_id' in eval_users_df.columns else eval_users_df.columns[0]
eval_users = eval_users_df[USER_COL].astype('int64').to_numpy()
eval_user_set = set(map(int, eval_users))

contact_df = pd.read_csv(CONTACT_EIDS_PATH)
CONTACT_COL = 'mapped_eid' if 'mapped_eid' in contact_df.columns else contact_df.columns[0]
contact_eids = set(contact_df[CONTACT_COL].astype(int).tolist())

print('eval users:', len(eval_users))
print('contact eids:', sorted(contact_eids), 'count:', len(contact_eids))


In [ ]:
# =========================
# 5. Streaming example builder
# =========================
def iter_parquet_batches(paths: List[Path], batch_rows: int):
    for path in paths:
        pf = pq.ParquetFile(path)
        missing = set(EVENT_COLUMNS) - set(pf.schema.names)
        if missing:
            raise ValueError(f'{path} missing columns {missing}')
        for batch in pf.iter_batches(batch_size=batch_rows, columns=EVENT_COLUMNS):
            df = batch.to_pandas()
            yield path, df
            del df
            gc.collect()

class ExampleStore:
    def __init__(self, max_examples: int, seq_len: int):
        self.max_examples = max_examples
        self.seq_len = seq_len
        self.x_items = np.zeros((max_examples, seq_len), dtype=np.int32)
        self.x_eids = np.zeros((max_examples, seq_len), dtype=np.int16)
        self.y = np.zeros(max_examples, dtype=np.int32)
        self.n = 0

    def add(self, history, pos_row: int):
        if self.n >= self.max_examples or not history:
            return
        hist = list(history)[-self.seq_len:]
        off = self.seq_len - len(hist)
        self.x_items[self.n, off:] = [x[0] for x in hist]
        self.x_eids[self.n, off:] = [x[1] for x in hist]
        self.y[self.n] = pos_row
        self.n += 1

    def finish(self):
        return self.x_items[:self.n], self.x_eids[:self.n], self.y[:self.n]

examples = ExampleStore(MAX_TRAIN_POSITIVES, MAX_SEQ_LEN)
item_counts = Counter()
eid_counts = Counter()
contact_item_counts = Counter()
feature_contact_counts = {c: Counter() for c in feature_cols}

# Eval histories are needed for prediction; train histories are processed per user and discarded.
eval_histories: Dict[int, deque] = {int(u): deque(maxlen=MAX_SEQ_LEN) for u in eval_users}

def update_stats(row: int, eid: int):
    item_counts[row] += 1
    eid_counts[eid] += 1
    if eid in contact_eids:
        contact_item_counts[row] += 1
        idx = row - 1
        for col in feature_cols:
            feature_contact_counts[col][int(raw_feature_arrays[col][idx])] += 1

def process_sorted_user_events(df: pd.DataFrame, keep_eval_histories: bool):
    rows = map_item_ids_to_rows(df['item_id'].to_numpy(np.int64))
    df = df.assign(item_row=rows)
    df = df[(df['item_row'] > 0) & valid_item_mask[df['item_row'].to_numpy(np.int32)]]
    if len(df) == 0:
        return 0
    df = df.sort_values(['user_id', 'timestamp'])
    last_user = None
    hist = deque(maxlen=MAX_SEQ_LEN)
    used = 0
    for user_id, eid, row in df[['user_id', 'eid', 'item_row']].itertuples(index=False, name=None):
        user_id = int(user_id); eid = int(eid); row = int(row)
        if user_id != last_user:
            hist = eval_histories[user_id] if keep_eval_histories and user_id in eval_histories else deque(maxlen=MAX_SEQ_LEN)
            last_user = user_id
        update_stats(row, eid)
        if eid in contact_eids and len(hist) > 0:
            examples.add(hist, row)
        hist.append((row, min(eid, 32767)))
        used += 1
    return used


In [ ]:
# =========================
# 6. Stream train partitions, then eval histories
# =========================
print('Scanning train partitions for neural next-contact examples...')
train_used = 0
for path, df in tqdm(iter_parquet_batches(TRAIN_PATHS, TRAIN_BATCH_ROWS), desc='train parquet batches'):
    if examples.n >= MAX_TRAIN_POSITIVES:
        break
    train_used += process_sorted_user_events(df, keep_eval_histories=False)
    del df
    gc.collect()

print('Scanning eval_user_events for histories and extra examples...')
eval_used = 0
for path, df in tqdm(iter_parquet_batches([EVAL_EVENTS_PATH], EVAL_BATCH_ROWS), desc='eval parquet batches'):
    # eval_user_events already contains eval users, but keep this as a safety filter.
    df = df[df['user_id'].isin(eval_user_set)]
    eval_used += process_sorted_user_events(df, keep_eval_histories=True)
    del df
    gc.collect()

X_items, X_eids, y = examples.finish()
print('train_used_events:', train_used)
print('eval_used_events:', eval_used)
print('training positives:', len(y))
print('arrays MB:', round((X_items.nbytes + X_eids.nbytes + y.nbytes) / 1024**2, 1))
print('top eids:', eid_counts.most_common(12))

if len(y) < 10_000:
    raise RuntimeError('Too few training examples; increase TRAIN_PARTITION_LIMIT or check contact_eids.')

del examples
gc.collect()


In [ ]:
# =========================
# 7. EDA and feature-selection report
# =========================
summary = {
    'eval_users': int(len(eval_users)),
    'item_rows': int(len(row_to_item_id) - 1),
    'feature_cols': ','.join(feature_cols),
    'train_partitions_scanned': int(len(TRAIN_PATHS)),
    'train_used_events': int(train_used),
    'eval_used_events': int(eval_used),
    'training_positives': int(len(y)),
    'max_seq_len': int(MAX_SEQ_LEN),
    'candidate_pool_cap': int(PER_USER_MAX_CANDIDATES),
}
pd.Series(summary).to_csv(REPORT_DIR / 'run_summary.csv')
print(json.dumps(summary, ensure_ascii=False, indent=2))

if plt is not None:
    fig_dir = REPORT_DIR / 'figures'
    fig_dir.mkdir(exist_ok=True)
    pd.Series(dict(eid_counts)).sort_values(ascending=False).head(30).sort_values().plot(kind='barh', figsize=(8, 7), color='#4d6a8a')
    plt.title('Top event ids used by the neural pipeline')
    plt.tight_layout(); plt.savefig(fig_dir / 'event_id_frequency.png', dpi=150); plt.close()
    pd.Series(dict(contact_item_counts)).sort_values(ascending=False).head(50).plot(kind='line', figsize=(8, 4), color='#b23a48')
    plt.title('Top contact-item frequency tail')
    plt.tight_layout(); plt.savefig(fig_dir / 'contact_item_tail.png', dpi=150); plt.close()


In [ ]:
# =========================
# 8. Dataset and the single neural model
# =========================
class NextContactDataset(Dataset):
    def __init__(self, x_items, x_eids, y, n_items: int, negatives: int, seed: int):
        self.x_items = x_items
        self.x_eids = x_eids
        self.y = y
        self.n_items = n_items
        self.negatives = negatives
        self.rng = np.random.default_rng(seed)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        pos = int(self.y[idx])
        neg = self.rng.integers(1, self.n_items + 1, size=self.negatives, dtype=np.int64)
        neg[neg == pos] = (pos % self.n_items) + 1
        return {
            'seq_items': torch.from_numpy(self.x_items[idx].astype(np.int64, copy=False)),
            'seq_eids': torch.from_numpy(self.x_eids[idx].astype(np.int64, copy=False)),
            'mask': torch.from_numpy((self.x_items[idx] > 0).astype(np.float32, copy=False)),
            'pos_item': torch.tensor(pos, dtype=torch.long),
            'neg_items': torch.from_numpy(neg),
        }

class AvitoOneNeuralRecommender(nn.Module):
    def __init__(self, cardinalities, item_feature_matrix, emb_dim=112, hidden_dim=176, max_eid=512, max_seq_len=48):
        super().__init__()
        self.register_buffer('item_feature_matrix', torch.as_tensor(item_feature_matrix, dtype=torch.long))
        self.feature_embeddings = nn.ModuleList([nn.Embedding(int(card) + 1, emb_dim, padding_idx=0) for card in cardinalities])
        self.item_norm = nn.LayerNorm(emb_dim)
        self.item_proj = nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.GELU(), nn.LayerNorm(emb_dim))
        self.eid_embedding = nn.Embedding(int(max_eid) + 1, emb_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_seq_len, emb_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=4, dim_feedforward=hidden_dim * 4,
            dropout=0.1, activation='gelu', batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.user_proj = nn.Sequential(nn.LayerNorm(emb_dim), nn.Linear(emb_dim, emb_dim), nn.Tanh())
        self.item_bias = nn.Embedding(item_feature_matrix.shape[0], 1)
        self.temperature = nn.Parameter(torch.tensor(0.07))
    def item_encode(self, item_rows):
        flat = item_rows.reshape(-1).clamp_min(0)
        feats = self.item_feature_matrix[flat]
        emb = 0
        for j, layer in enumerate(self.feature_embeddings):
            emb = emb + layer(feats[:, j])
        emb = self.item_proj(self.item_norm(emb))
        return emb.reshape(*item_rows.shape, -1)
    def user_encode(self, seq_items, seq_eids, mask):
        item_emb = self.item_encode(seq_items.clamp_min(0))
        eid_emb = self.eid_embedding(seq_eids.clamp(0, self.eid_embedding.num_embeddings - 1))
        pos = torch.arange(seq_items.shape[1], device=seq_items.device).unsqueeze(0)
        x = (item_emb + eid_emb + self.pos_embedding(pos)) * mask.unsqueeze(-1)
        key_padding = mask.eq(0)
        empty = mask.sum(1).eq(0)
        if empty.any():
            key_padding = key_padding.clone(); key_padding[empty] = False
        enc = self.encoder(x, src_key_padding_mask=key_padding)
        pooled = (enc * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1.0)
        return self.user_proj(pooled)
    def score_items(self, user_vec, item_rows):
        item_vec = self.item_encode(item_rows)
        scores = (user_vec.unsqueeze(-2) * item_vec).sum(-1)
        scores = scores / self.temperature.abs().clamp_min(0.02)
        return scores + self.item_bias(item_rows).squeeze(-1)
    def forward(self, batch):
        user_vec = self.user_encode(batch['seq_items'], batch['seq_eids'], batch['mask'])
        candidates = torch.cat([batch['pos_item'].unsqueeze(1), batch['neg_items']], dim=1)
        return self.score_items(user_vec, candidates)


In [ ]:
# =========================
# 9. Train the neural model
# =========================
n_items = len(row_to_item_id) - 1
max_eid = max(max(eid_counts.keys()) if eid_counts else 1, max(contact_eids) if contact_eids else 1)
perm = np.random.default_rng(SEED).permutation(len(y))
X_items = X_items[perm]; X_eids = X_eids[perm]; y = y[perm]

dataset = NextContactDataset(X_items, X_eids, y, n_items=n_items, negatives=NEGATIVES, seed=SEED)
val_size = max(1, min(len(dataset) // 20, 50_000))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')

model = AvitoOneNeuralRecommender(cardinalities, feature_matrix, EMB_DIM, HIDDEN_DIM, max_eid, MAX_SEQ_LEN).to(DEVICE)
if USE_DATA_PARALLEL and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')
best_val = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train(); losses = []
    for batch in tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}'):
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        labels = torch.zeros(batch['pos_item'].shape[0], dtype=torch.long, device=DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            logits = model(batch)
            loss = F.cross_entropy(logits, labels)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        losses.append(float(loss.detach().cpu()))
    model.eval(); val_losses=[]; val_top1=[]
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            labels = torch.zeros(batch['pos_item'].shape[0], dtype=torch.long, device=DEVICE)
            logits = model(batch)
            val_losses.append(float(F.cross_entropy(logits, labels).cpu()))
            val_top1.append(float((logits.argmax(1) == 0).float().mean().cpu()))
    val_loss = float(np.mean(val_losses)); top1 = float(np.mean(val_top1))
    print(f'epoch={epoch} train_loss={np.mean(losses):.5f} val_loss={val_loss:.5f} val_top1={top1:.5f}')
    if val_loss < best_val:
        best_val = val_loss
        torch.save((model.module if isinstance(model, nn.DataParallel) else model).state_dict(), MODEL_DIR / 'model.pt')

base_model = model.module if isinstance(model, nn.DataParallel) else model
base_model.load_state_dict(torch.load(MODEL_DIR / 'model.pt', map_location=DEVICE))
base_model.eval()
print('best_val_loss:', best_val)


In [ ]:
# =========================
# 10. Smart candidate maps from selected features
# =========================
def top_rows_from_counter(counter: Counter, limit: int) -> List[int]:
    rows = []
    for row, _ in counter.most_common(limit * 2):
        if 0 < row < len(valid_item_mask) and valid_item_mask[row]:
            rows.append(int(row))
            if len(rows) >= limit:
                break
    return rows

global_candidates = top_rows_from_counter(contact_item_counts if contact_item_counts else item_counts, GLOBAL_CANDIDATES)
if len(global_candidates) < GLOBAL_CANDIDATES:
    global_candidates += top_rows_from_counter(item_counts, GLOBAL_CANDIDATES - len(global_candidates))

feature_top_rows = {col: {} for col in feature_cols}
# Build top rows per selected item feature using observed item/contact popularity.
base_counter = contact_item_counts if contact_item_counts else item_counts
for col in feature_cols:
    buckets = defaultdict(list)
    arr = raw_feature_arrays[col]
    for row, cnt in base_counter.most_common(250_000):
        if row <= 0 or row >= len(valid_item_mask) or not valid_item_mask[row]:
            continue
        buckets[int(arr[row - 1])].append((int(row), int(cnt)))
    for val, pairs in buckets.items():
        feature_top_rows[col][val] = [r for r, _ in pairs[:PER_FEATURE_CANDIDATES]]

print('global candidates:', len(global_candidates))
print('feature maps:', {c: len(feature_top_rows[c]) for c in feature_cols})


In [ ]:
# =========================
# 11. Predict with neural reranking and save submission.csv
# =========================
def user_history_tensors(user_ids_batch):
    seq_i = np.zeros((len(user_ids_batch), MAX_SEQ_LEN), dtype=np.int64)
    seq_e = np.zeros((len(user_ids_batch), MAX_SEQ_LEN), dtype=np.int64)
    masks = np.zeros((len(user_ids_batch), MAX_SEQ_LEN), dtype=np.float32)
    seen_sets = []
    profile_values = []
    for n, user_id in enumerate(user_ids_batch):
        hist = list(eval_histories.get(int(user_id), []))[-MAX_SEQ_LEN:]
        seen = {int(x[0]) for x in eval_histories.get(int(user_id), [])}
        seen_sets.append(seen)
        vals = {c: Counter() for c in feature_cols}
        if hist:
            off = MAX_SEQ_LEN - len(hist)
            rows = [x[0] for x in hist]
            seq_i[n, off:] = rows
            seq_e[n, off:] = [x[1] for x in hist]
            masks[n, off:] = 1.0
            for row in rows[-24:]:
                idx = row - 1
                for c in feature_cols:
                    vals[c][int(raw_feature_arrays[c][idx])] += 1
        profile_values.append(vals)
    return seq_i, seq_e, masks, seen_sets, profile_values

def build_user_candidates(profile_vals, seen_rows):
    cand = []
    cand.extend(global_candidates)
    # Prioritize vertical/category/sid matches from the recent history.
    for col in feature_cols:
        for val, _ in profile_vals[col].most_common(3):
            cand.extend(feature_top_rows[col].get(int(val), []))
    # Deduplicate, filter seen, cap.
    out = []
    used = set()
    for r in cand:
        if r in used or r in seen_rows or r <= 0 or r >= len(valid_item_mask) or not valid_item_mask[r]:
            continue
        out.append(int(r)); used.add(int(r))
        if len(out) >= PER_USER_MAX_CANDIDATES:
            break
    # Ensure at least K candidates even for cold histories.
    if len(out) < SUBMISSION_K:
        for r in global_candidates:
            if r not in used and r not in seen_rows:
                out.append(int(r)); used.add(int(r))
                if len(out) >= SUBMISSION_K:
                    break
    return np.asarray(out, dtype=np.int64)

rows_out = []
with torch.no_grad():
    for start in tqdm(range(0, len(eval_users), BATCH_SIZE), desc='predict users'):
        users_b = eval_users[start:start + BATCH_SIZE]
        seq_i, seq_e, masks, seen_sets, profiles = user_history_tensors(users_b)
        seq_i_t = torch.tensor(seq_i, dtype=torch.long, device=DEVICE)
        seq_e_t = torch.tensor(seq_e, dtype=torch.long, device=DEVICE)
        masks_t = torch.tensor(masks, dtype=torch.float32, device=DEVICE)
        user_vec = base_model.user_encode(seq_i_t, seq_e_t, masks_t)
        for i, user_id in enumerate(users_b):
            cand_np = build_user_candidates(profiles[i], seen_sets[i])
            if len(cand_np) < SUBMISSION_K:
                raise RuntimeError(f'user {user_id} has only {len(cand_np)} candidates')
            cand_t = torch.tensor(cand_np, dtype=torch.long, device=DEVICE)
            scores = base_model.score_items(user_vec[i:i+1], cand_t.unsqueeze(0)).squeeze(0)
            top_idx = torch.topk(scores, k=SUBMISSION_K).indices.detach().cpu().numpy()
            rec_rows = cand_np[top_idx]
            rows_out.extend((int(user_id), int(row_to_item_id[r])) for r in rec_rows)

submission = pd.DataFrame(rows_out, columns=['user_id', 'item_id']).drop_duplicates(['user_id', 'item_id'])
counts = submission.groupby('user_id').size()
print('submission shape:', submission.shape)
print('users:', submission['user_id'].nunique(), 'min_k:', int(counts.min()), 'max_k:', int(counts.max()))
assert list(submission.columns) == ['user_id', 'item_id']
assert submission.duplicated(['user_id', 'item_id']).sum() == 0
assert submission['user_id'].nunique() == len(eval_users)
assert counts.max() <= SUBMISSION_K
assert counts.min() == SUBMISSION_K
submission.to_csv(SUBMISSION_FILE, index=False)
print('saved:', SUBMISSION_FILE)


## Neural-only claim

There is one trainable model in this notebook: `AvitoOneNeuralRecommender`. It encodes user sequences and item metadata end-to-end and produces final scores for the submitted ranking. Candidate generation is a deterministic memory-control stage based on official item features and observed pre-cutoff events; it is not a second model, not a tree model, and not an ensemble.